<a href="https://colab.research.google.com/github/icsl-aist/hsr-genesis/blob/main/examples/tutorials/5_grasp_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HSR 把持の全シーケンス / HSR Full Grasp Sequence

**目的 / Objective:**

- ベース移動・アームIK・グリッパ制御を組み合わせた **ピック＆プレース** の一連の流れを学ぶ / Learn the complete pick-and-place workflow combining base motion, arm IK, and gripper control.
- アプローチ → ハンド開 → 把持 → 持ち上げ → 移動 → 離す → 退避 の一連の流れを体験する / Experience the full sequence: approach → open hand → grasp → lift → move → release → retreat.

> **Note:** Run this on a GPU runtime (Colab: *Runtime → Change runtime type → T4 GPU* or better).

## Setup / セットアップ

Run the cell below to install dependencies, clone the repo, and configure GPU rendering.
If you run into issues, see the [troubleshoot notebook](7_troubleshoot_colab.ipynb).

下のセルを実行して、依存パッケージのインストール、リポジトリのクローン、GPU レンダリングの設定を行います。
問題が発生した場合は[トラブルシューティングノートブック](7_troubleshoot_colab.ipynb)を参照してください。


In [ ]:
import importlib, urllib.request

# Fetch standalone bootstrap from GitHub (zero hsr_genesis imports).
exec(urllib.request.urlopen(
    "https://raw.githubusercontent.com/icsl-aist/hsr-genesis/main/colab_setup.py"
).read())

# One-call setup: installs deps, clones repo (with submodules),
# configures EGL for headless GPU rendering. Safe to re-run.
# See the troubleshoot notebook if anything goes wrong.
setup_colab()


In [ ]:
from hsr_genesis.tutorial_utils import *

init_sim()


## 5. 把持シーケンスの概要 / Grasp sequence overview

ピック＆プレースは以下のステップで構成されます / The pick-and-place consists of the following steps:

1. **アプローチ / Approach** — アームをニュートラルにし、ハンドを開く / Move arm to neutral and open the hand.
2. **把持位置へ / Pre-grasp** — IKでハンドを物体の上に近づける / Bring the hand above the object with IK.
3. **把持 / Grasp** — 力制御でグリッパを閉じる / Close the gripper under force control.
4. **持ち上げ / Lift** — 把持したままハンドを持ち上げる / Lift the hand while keeping the grasp.
5. **移動 / Move base** — ベースを別の場所へ移動させる / Move the base to a new location.
6. **降ろして離す / Lower & release** — ハンドを下げて物体を離す / Lower the hand and release the object.
7. **退避 / Retreat** — アームをニュートラルに戻す / Return the arm to neutral.

まず対象物を配置します / First, spawn the target object.

## 6. 対象物を配置 / Spawn target object

In [ ]:
cube = spawn_box((0.45, 0.0, 0.02), color=(0.8, 0.2, 0.2, 1.0))
run(0.5)
show_video()

## 7. Step 1 — プレ把持姿勢 / Move to pre-grasp

アームをニュートラルにし、ハンドを開いておきます。
Move the arm to neutral and open the hand in preparation.

In [ ]:
move_arm_neutral()
move_hand(1.0)
run(2.0)
show_video()

## 8. Step 2 — IKでアプローチ / Approach with IK

IKでハンドを物体の高さ (z=0.02 m) に移動します。ロール 180° で指が下を向きます。
Use IK to move the hand to the object height (z=0.02 m). A roll of 180° points the fingers downward.

In [ ]:
move_hand(1.0)
move_wholebody_ik(0.45, 0.0, 0.02, 180, 0, 0)
run(3.0)
show_video()

## 9. Step 3 — 把持 / Grasp

In [ ]:
grasp_object(5.0)
run(3.0)
show_video()

## 10. Step 4 — 持ち上げ / Lift

把持したままハンドを z=0.35 m まで持ち上げます。
Lift the hand to z=0.35 m while keeping the grasp.

In [ ]:
move_wholebody_ik(0.45, 0.0, 0.35, 180, 0, 0)
run(2.0)
show_video()

## 11. Step 5 — ベースを移動 / Move base to a new location

`move_base_goal(x, y, theta)` でベースを新しい位置へ移動させます。把持は `run()` 中も維持されます。
Move the base to a new position with `move_base_goal(x, y, theta)`. The grasp is maintained during `run()`.

In [ ]:
move_base_goal(0.0, 0.3, 30)
run(3.0)
show_video()

## 12. Step 6 — 降ろして離す / Lower and release

IKでハンドを下げ、`release_object()` で把持を解除します。
Lower the hand with IK and release the object with `release_object()`.

In [ ]:
move_wholebody_ik(0.0, 0.3, 0.1, 180, 0, 0)
run(2.0)
release_object()
run(1.0)
show_video()

## 13. Step 7 — 退避 / Retreat

アームをニュートラルに戻して安全な姿勢にします。
Return the arm to neutral for a safe pose.

In [ ]:
move_arm_neutral()
run(2.0)
show_video()

## まとめ / Summary

ピック＆プレースの一連の流れを振り返ります / Review of the full pick-and-place pipeline:

| Step | 関数 / Function | 説明 / Description |
|------|-----------------|---------------------|
| Pre-grasp | `move_arm_neutral()`, `move_hand(1.0)` | アームをニュートラル・ハンドを開く / Neutral arm, open hand |
| Approach | `move_wholebody_ik(...)` | IKで物体の上へ / IK to above the object |
| Grasp | `grasp_object(effort)` | 力制御で把持 / Force-controlled grasp |
| Lift | `move_wholebody_ik(...)` | 把持したまま持ち上げ / Lift while grasping |
| Move | `move_base_goal(x, y, theta)` | ベースを移動 / Move the base |
| Release | `move_wholebody_ik(...)`, `release_object()` | 降ろして離す / Lower and release |
| Retreat | `move_arm_neutral()` | 退避 / Return to neutral |

これでベース・アーム・グリッパを組み合わせた一連の操作ができるようになりました。
You can now combine base, arm, and gripper into a complete manipulation sequence.